# 02 - CafeBERT base model: 5-fold OOF + full-train refit (seed=random)

Outputs `artifacts/probs/cafebert_{oof,val,test}.npy` and `artifacts/metrics/cafebert.json`.

Hyperparameters are aligned with `EmoModel_Advanced_CafeBert_Test.ipynb` while keeping 5-fold OOF.

In [ ]:
%pip install -q transformers datasets accelerate scikit-learn pandas numpy sentencepiece

In [ ]:
import sys, os

REPO_ROOT = '/content/drive/MyDrive/thesis/topicmodeling'
TMP_ROOT = '/content/vsfc_ensemble_tmp'

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
else:
    import pathlib
    LOCAL = pathlib.Path.cwd().resolve()
    while LOCAL.name != 'tm_research' and LOCAL.parent != LOCAL:
        LOCAL = LOCAL.parent
    sys.path.insert(0, str(LOCAL.parent))

from tm_research.VSFC_ensemble.colab_setup import setup_colab
paths = setup_colab(repo_root=REPO_ROOT, tmp_root=TMP_ROOT)

In [ ]:
from tm_research.VSFC_ensemble.utils_io import load_vsfc_splits
from tm_research.VSFC_ensemble.bert_oof import BertOOFConfig, run_bert_oof

train_df, val_df, test_df, label_map = load_vsfc_splits()
print('train', train_df.shape, 'val', val_df.shape, 'test', test_df.shape)
print('classes', label_map.class_names)

In [ ]:
import random

run_seed = random.SystemRandom().randint(1, 1_000_000)
print(f'Using random seed: {run_seed}')

cfg = BertOOFConfig(
    model_name='uitnlp/CafeBERT',
    output_name='cafebert',
    seed=run_seed,
    n_folds=3,
    num_epochs=20,
    learning_rate=2e-5,
    max_length=256,
    train_batch_size=32,
    eval_batch_size=64,
    grad_accum_steps=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    early_stopping_patience=5,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,
    work_dir=str(paths.bert_work / 'cafebert'),
)
metrics = run_bert_oof(cfg, train_df, val_df, test_df, label_map)
metrics

In [ ]:
from tm_research.VSFC_ensemble.utils_io import push_artifacts_to_persistent
push_artifacts_to_persistent()